# Notebook 02: County-Year Feature Dataset Construction

This notebook converts the cleaned monthly Zillow–FEMA panel into the final
Florida county-year feature dataset for 2011–2025. It constructs housing,
spatial, disaster-history, socioeconomic, affordability, and NFIP
insurance-loss indicators and saves the integrated dataset used in the
remaining analysis.


## 1. Setup and Input Validation

The default workflow uses the frozen ACS, county-adjacency, and NFIP snapshots
retained for this study. The refresh switches can be enabled when updated
external data are intentionally required.


In [10]:
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start_path=None):
    # Locate the repository root from the current directory or its parents.
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "notebooks").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root()
RAW_DATA = PROJECT_ROOT / "data" / "raw"
SOURCE_PREPARATION_DIR = (
    PROJECT_ROOT / "data" / "interim" / "source_preparation"
)
COUNTY_YEAR_DIR = PROJECT_ROOT / "data" / "interim" / "county_year"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
RESULTS_TABLES = PROJECT_ROOT / "results" / "tables"

for directory in (COUNTY_YEAR_DIR, PROCESSED_DATA, RESULTS_TABLES):
    directory.mkdir(parents=True, exist_ok=True)

STUDY_START_YEAR = 2011
STUDY_END_YEAR = 2025
ACS_START_YEAR = 2011
ACS_END_YEAR = 2024
RANDOM_STATE = 42

REFRESH_SPATIAL_ADJACENCY = False
REFRESH_ACS_DATA = False
REFRESH_NFIP_DATA = False

MONTHLY_INPUT_PATH = (
    SOURCE_PREPARATION_DIR / "merged_florida_zillow_fema_monthly.csv"
)
FINAL_FEATURE_PATH = (
    PROCESSED_DATA / "florida_county_year_features_2011_2025.csv"
)

if not MONTHLY_INPUT_PATH.exists():
    raise FileNotFoundError(
        "Run cleaned Notebook 01 first. Missing input: "
        f"{MONTHLY_INPUT_PATH}"
    )

monthly_data = pd.read_csv(MONTHLY_INPUT_PATH)
monthly_data["Date"] = pd.to_datetime(monthly_data["Date"])

monthly_validation = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Columns",
            "Florida counties",
            "Start date",
            "End date",
            "Duplicate county-months",
            "Missing housing values",
        ],
        "value": [
            len(monthly_data),
            monthly_data.shape[1],
            monthly_data["STCOFIPS"].nunique(),
            monthly_data["Date"].min().date(),
            monthly_data["Date"].max().date(),
            int(monthly_data.duplicated(["STCOFIPS", "Date"]).sum()),
            int(monthly_data["HousingPrice"].isna().sum()),
        ],
    }
)

display(monthly_validation)


,check,value
0,Rows,12767
1,Columns,18
2,Florida counties,67
3,Start date,2010-01-31
4,End date,2025-12-31
5,Duplicate county-months,0
6,Missing housing values,0


## 2. County-Year Housing Indicators

Monthly housing observations are aggregated to annual county values. Growth,
baseline appreciation, volatility, acceleration, and market-signal flags are
calculated before the panel is restricted to the 2011–2025 study period. The
2010 observations are retained only to establish baselines and first-year
growth values.


In [11]:
monthly_data["Year"] = monthly_data["Date"].dt.year
monthly_data["Metro"] = monthly_data["Metro"].fillna("Non-Metro")
monthly_data["is_metro"] = monthly_data["Metro"].ne("Non-Metro").astype(int)

HOUSING_IDENTIFIER_COLUMNS = [
    "STCOFIPS",
    "RegionID",
    "RegionName",
    "State",
    "Metro",
    "StateCodeFIPS",
    "MunicipalCodeFIPS",
    "COUNTY",
]

county_year_housing = (
    monthly_data.groupby(
        HOUSING_IDENTIFIER_COLUMNS + ["Year"], as_index=False
    )
    .agg(
        avg_annual_housing_price=("HousingPrice", "mean"),
        median_annual_housing_price=("HousingPrice", "median"),
        min_annual_housing_price=("HousingPrice", "min"),
        max_annual_housing_price=("HousingPrice", "max"),
        annual_price_volatility=("HousingPrice", "std"),
        monthly_observations=("HousingPrice", "count"),
        is_metro=("is_metro", "first"),
        SizeRank=("SizeRank", "first"),
        POPULATION=("POPULATION", "first"),
        CFLD_RISKS=("CFLD_RISKS", "first"),
        HRCN_RISKS=("HRCN_RISKS", "first"),
        SOVI_SCORE=("SOVI_SCORE", "first"),
        RESL_SCORE=("RESL_SCORE", "first"),
    )
)

county_year_housing["annual_price_volatility"] = (
    county_year_housing["annual_price_volatility"].fillna(0)
)

county_year_housing = (
    county_year_housing.sort_values(["STCOFIPS", "Year"])
    .reset_index(drop=True)
)

county_year_housing["prev_year_housing_price"] = (
    county_year_housing.groupby("STCOFIPS")["avg_annual_housing_price"]
    .shift(1)
)
county_year_housing["annual_price_growth_dollar"] = (
    county_year_housing["avg_annual_housing_price"]
    - county_year_housing["prev_year_housing_price"]
)
county_year_housing["annual_price_growth_pct"] = (
    county_year_housing["annual_price_growth_dollar"]
    / county_year_housing["prev_year_housing_price"]
    * 100
)
county_year_housing["baseline_housing_price"] = (
    county_year_housing.groupby("STCOFIPS")["avg_annual_housing_price"]
    .transform("first")
)
county_year_housing["appreciation_from_baseline_pct"] = (
    (
        county_year_housing["avg_annual_housing_price"]
        - county_year_housing["baseline_housing_price"]
    )
    / county_year_housing["baseline_housing_price"]
    * 100
)
county_year_housing["price_growth_acceleration"] = (
    county_year_housing.groupby("STCOFIPS")["annual_price_growth_pct"]
    .diff()
)


In [12]:
high_growth_threshold = county_year_housing["annual_price_growth_pct"].median()
high_volatility_threshold = county_year_housing[
    "annual_price_volatility"
].median()

county_year_housing["high_growth_flag"] = np.where(
    county_year_housing["annual_price_growth_pct"].isna(),
    np.nan,
    np.where(
        county_year_housing["annual_price_growth_pct"]
        >= high_growth_threshold,
        1,
        0,
    ),
)
county_year_housing["high_volatility_flag"] = np.where(
    county_year_housing["annual_price_volatility"]
    >= high_volatility_threshold,
    1,
    0,
)
county_year_housing["complete_year_flag"] = (
    county_year_housing["monthly_observations"].eq(12).astype(int)
)

county_year_housing_final = county_year_housing.loc[
    county_year_housing["Year"].between(
        STUDY_START_YEAR, STUDY_END_YEAR
    )
].copy()

housing_duplicate_count = int(
    county_year_housing_final.duplicated(["STCOFIPS", "Year"]).sum()
)

housing_validation = pd.DataFrame(
    {
        "check": [
            "County-year rows",
            "Variables",
            "Florida counties",
            "Start year",
            "End year",
            "Duplicate county-years",
            "Complete county-years",
            "High-growth threshold",
            "High-volatility threshold",
        ],
        "value": [
            len(county_year_housing_final),
            county_year_housing_final.shape[1],
            county_year_housing_final["STCOFIPS"].nunique(),
            county_year_housing_final["Year"].min(),
            county_year_housing_final["Year"].max(),
            housing_duplicate_count,
            int(county_year_housing_final["complete_year_flag"].sum()),
            high_growth_threshold,
            high_volatility_threshold,
        ],
    }
)

if len(county_year_housing_final) != 1000:
    raise ValueError("Expected 1,000 county-year housing observations.")
if housing_duplicate_count:
    raise ValueError("Duplicate county-year housing observations found.")

HOUSING_OUTPUT_PATH = COUNTY_YEAR_DIR / "county_year_housing_indicators.csv"
county_year_housing_final.to_csv(HOUSING_OUTPUT_PATH, index=False)

county_year_overview = housing_validation.iloc[:7].copy()
county_year_overview.columns = ["Metric", "Value"]
county_year_overview.to_csv(
    RESULTS_TABLES / "county_year_housing_overview.csv", index=False
)

annual_housing_summary = (
    county_year_housing_final.groupby("Year", as_index=False)
    .agg(
        avg_florida_housing_price=("avg_annual_housing_price", "mean"),
        median_florida_housing_price=("avg_annual_housing_price", "median"),
        avg_annual_growth_pct=("annual_price_growth_pct", "mean"),
        avg_annual_volatility=("annual_price_volatility", "mean"),
        county_count=("STCOFIPS", "nunique"),
    )
    .round(2)
)
annual_housing_summary.to_csv(
    RESULTS_TABLES / "annual_housing_summary.csv", index=False
)

display(housing_validation)


,check,value
0,County-year rows,1000.000000
1,Variables,31.000000
2,Florida counties,67.000000
3,Start year,2011.000000
4,End year,2025.000000
5,Duplicate county-years,0.000000
6,Complete county-years,999.000000
7,High-growth threshold,5.916556
8,High-volatility threshold,3724.468214


## 3. Spatial Adjacency and Neighbouring-County Features

County adjacency is based on the 2025 TIGER/Line county boundaries. The frozen
adjacency table is used by default; enabling the refresh switch reconstructs it
from the shapefile. Neighbouring housing indicators are then calculated for
each county-year.


In [13]:
ADJACENCY_PATH = COUNTY_YEAR_DIR / "florida_county_adjacency.csv"
SHAPEFILE_PATH = (
    RAW_DATA
    / "shapefiles"
    / "tl_2025_us_county"
    / "tl_2025_us_county.shp"
)

if REFRESH_SPATIAL_ADJACENCY:
    import geopandas as gpd

    county_shapes = gpd.read_file(SHAPEFILE_PATH)
    florida_shapes = county_shapes.loc[
        county_shapes["STATEFP"].eq("12")
    ].copy()
    florida_shapes["STCOFIPS"] = florida_shapes["GEOID"].astype(str)
    county_geometries = florida_shapes[
        ["STCOFIPS", "NAME", "NAMELSAD", "geometry"]
    ].copy()

    touching = gpd.sjoin(
        county_geometries,
        county_geometries,
        how="inner",
        predicate="touches",
        lsuffix="county",
        rsuffix="neighbor",
    )
    touching = touching.loc[
        touching["STCOFIPS_county"]
        != touching["STCOFIPS_neighbor"]
    ]
    county_adjacency = (
        touching[
            [
                "STCOFIPS_county",
                "NAME_county",
                "STCOFIPS_neighbor",
                "NAME_neighbor",
            ]
        ]
        .rename(
            columns={
                "STCOFIPS_county": "county_fips",
                "NAME_county": "county_name",
                "STCOFIPS_neighbor": "neighbor_fips",
                "NAME_neighbor": "neighbor_name",
            }
        )
        .sort_values(["county_fips", "neighbor_fips"])
        .reset_index(drop=True)
    )
    county_adjacency.to_csv(ADJACENCY_PATH, index=False)
else:
    if not ADJACENCY_PATH.exists():
        raise FileNotFoundError(
            "County-adjacency snapshot is missing. Enable "
            "REFRESH_SPATIAL_ADJACENCY to rebuild it."
        )
    county_adjacency = pd.read_csv(
        ADJACENCY_PATH,
        dtype={"county_fips": str, "neighbor_fips": str},
    )

county_adjacency["county_fips"] = (
    county_adjacency["county_fips"].str.zfill(5)
)
county_adjacency["neighbor_fips"] = (
    county_adjacency["neighbor_fips"].str.zfill(5)
)

if county_adjacency.duplicated(
    ["county_fips", "neighbor_fips"]
).any():
    raise ValueError("Duplicate county-neighbour pairs found.")

print(
    f"Adjacency pairs: {len(county_adjacency)}; "
    f"counties: {county_adjacency['county_fips'].nunique()}"
)


Adjacency pairs: 320; counties: 67


In [14]:
county_year_housing_final["STCOFIPS"] = (
    county_year_housing_final["STCOFIPS"].astype(str).str.zfill(5)
)

neighbor_housing_base = county_year_housing_final[
    [
        "STCOFIPS",
        "Year",
        "avg_annual_housing_price",
        "annual_price_growth_pct",
        "annual_price_volatility",
        "high_growth_flag",
        "high_volatility_flag",
    ]
].rename(
    columns={
        "STCOFIPS": "neighbor_fips",
        "avg_annual_housing_price": "neighbor_housing_price",
        "annual_price_growth_pct": "neighbor_price_growth_pct",
        "annual_price_volatility": "neighbor_price_volatility",
        "high_growth_flag": "neighbor_high_growth_flag",
        "high_volatility_flag": "neighbor_high_volatility_flag",
    }
)

county_neighbor_housing = county_adjacency.merge(
    neighbor_housing_base,
    on="neighbor_fips",
    how="left",
    validate="many_to_many",
)

neighbor_housing_features = (
    county_neighbor_housing.groupby(
        ["county_fips", "Year"], as_index=False
    )
    .agg(
        neighbor_count=("neighbor_fips", "nunique"),
        neighbor_avg_housing_price=("neighbor_housing_price", "mean"),
        neighbor_avg_price_growth_pct=("neighbor_price_growth_pct", "mean"),
        neighbor_avg_price_volatility=("neighbor_price_volatility", "mean"),
        neighbor_high_growth_share=("neighbor_high_growth_flag", "mean"),
        neighbor_high_volatility_share=(
            "neighbor_high_volatility_flag", "mean"
        ),
    )
    .rename(columns={"county_fips": "STCOFIPS"})
)

county_year_housing_spatial = county_year_housing_final.merge(
    neighbor_housing_features,
    on=["STCOFIPS", "Year"],
    how="left",
    validate="one_to_one",
)

SPATIAL_FEATURE_COLUMNS = [
    "neighbor_count",
    "neighbor_avg_housing_price",
    "neighbor_avg_price_growth_pct",
    "neighbor_avg_price_volatility",
    "neighbor_high_growth_share",
    "neighbor_high_volatility_share",
]

if county_year_housing_spatial[SPATIAL_FEATURE_COLUMNS].isna().any().any():
    raise ValueError("Missing neighbouring-county features found.")

SPATIAL_HOUSING_PATH = (
    COUNTY_YEAR_DIR / "county_year_housing_spatial_features.csv"
)
NEIGHBOR_FEATURE_PATH = COUNTY_YEAR_DIR / "neighbor_housing_features.csv"

county_year_housing_spatial.to_csv(SPATIAL_HOUSING_PATH, index=False)
neighbor_housing_features.to_csv(NEIGHBOR_FEATURE_PATH, index=False)

spatial_validation = pd.DataFrame(
    {
        "check": [
            "County-year rows",
            "Florida counties",
            "Adjacency pairs",
            "Minimum neighbours",
            "Maximum neighbours",
            "Missing spatial values",
        ],
        "value": [
            len(county_year_housing_spatial),
            county_year_housing_spatial["STCOFIPS"].nunique(),
            len(county_adjacency),
            county_year_housing_spatial["neighbor_count"].min(),
            county_year_housing_spatial["neighbor_count"].max(),
            int(
                county_year_housing_spatial[SPATIAL_FEATURE_COLUMNS]
                .isna()
                .sum()
                .sum()
            ),
        ],
    }
)

display(spatial_validation)


,check,value
0,County-year rows,1000
1,Florida counties,67
2,Adjacency pairs,320
3,Minimum neighbours,1
4,Maximum neighbours,10
5,Missing spatial values,0


## 4. FEMA Disaster-History Integration

Florida county declarations are restricted to 2011–2025 and classified into
the climate-related incident types used in the study. Annual, cumulative, and
recent three-year disaster indicators are aligned with the complete project
county-year grid; county-years without declarations receive zeros.


In [15]:
FEMA_DISASTER_PATH = (
    RAW_DATA
    / "fema_disaster_declarations"
    / "DisasterDeclarationsSummaries.csv"
)

fema_disasters = pd.read_csv(
    FEMA_DISASTER_PATH,
    dtype={
        "state": str,
        "fipsStateCode": str,
        "fipsCountyCode": str,
        "placeCode": str,
    },
    low_memory=False,
)

for date_column in [
    "declarationDate",
    "incidentBeginDate",
    "incidentEndDate",
    "disasterCloseoutDate",
    "lastIAFilingDate",
    "lastRefresh",
]:
    if date_column in fema_disasters.columns:
        fema_disasters[date_column] = pd.to_datetime(
            fema_disasters[date_column], errors="coerce"
        )

florida_disasters = fema_disasters.loc[
    fema_disasters["state"].eq("FL")
].copy()
florida_disasters["fipsStateCode"] = (
    florida_disasters["fipsStateCode"].str.zfill(2)
)
florida_disasters["fipsCountyCode"] = (
    florida_disasters["fipsCountyCode"].str.zfill(3)
)
florida_disasters["STCOFIPS"] = (
    florida_disasters["fipsStateCode"]
    + florida_disasters["fipsCountyCode"]
)
florida_disasters["Year"] = florida_disasters[
    "incidentBeginDate"
].dt.year

county_year_spatial = pd.read_csv(
    SPATIAL_HOUSING_PATH, dtype={"STCOFIPS": str}
)
county_year_spatial["STCOFIPS"] = (
    county_year_spatial["STCOFIPS"].str.zfill(5)
)

project_counties = set(county_year_spatial["STCOFIPS"].unique())
florida_disasters = florida_disasters.loc[
    florida_disasters["Year"].between(
        STUDY_START_YEAR, STUDY_END_YEAR
    )
    & florida_disasters["STCOFIPS"].isin(project_counties)
].copy()

CLIMATE_INCIDENT_TYPES = [
    "Hurricane",
    "Tropical Storm",
    "Severe Storm",
    "Flood",
    "Coastal Storm",
    "Fire",
    "Tornado",
    "Freezing",
]

florida_disasters["incidentType"] = (
    florida_disasters["incidentType"].astype(str).str.strip()
)
florida_disasters["climate_disaster_flag"] = (
    florida_disasters["incidentType"].isin(CLIMATE_INCIDENT_TYPES).astype(int)
)
for incident, output_column in {
    "Hurricane": "hurricane_disaster_flag",
    "Tropical Storm": "tropical_storm_disaster_flag",
    "Severe Storm": "severe_storm_disaster_flag",
    "Flood": "flood_disaster_flag",
    "Fire": "fire_disaster_flag",
}.items():
    florida_disasters[output_column] = (
        florida_disasters["incidentType"].eq(incident).astype(int)
    )


In [16]:
county_year_disasters = (
    florida_disasters.groupby(["STCOFIPS", "Year"])
    .agg(
        disaster_count=("disasterNumber", "count"),
        unique_disaster_events=("disasterNumber", "nunique"),
        climate_disaster_count=("climate_disaster_flag", "sum"),
        hurricane_disaster_count=("hurricane_disaster_flag", "sum"),
        tropical_storm_disaster_count=(
            "tropical_storm_disaster_flag", "sum"
        ),
        severe_storm_disaster_count=("severe_storm_disaster_flag", "sum"),
        flood_disaster_count=("flood_disaster_flag", "sum"),
        fire_disaster_count=("fire_disaster_flag", "sum"),
    )
    .reset_index()
)
county_year_disasters["any_disaster_flag"] = (
    county_year_disasters["disaster_count"].gt(0).astype(int)
)
county_year_disasters["any_climate_disaster_flag"] = (
    county_year_disasters["climate_disaster_count"].gt(0).astype(int)
)

county_year_grid = county_year_spatial[
    ["STCOFIPS", "Year"]
].drop_duplicates()
county_year_disasters = county_year_grid.merge(
    county_year_disasters,
    on=["STCOFIPS", "Year"],
    how="left",
    validate="one_to_one",
)

DISASTER_ANNUAL_COLUMNS = [
    "disaster_count",
    "unique_disaster_events",
    "climate_disaster_count",
    "hurricane_disaster_count",
    "tropical_storm_disaster_count",
    "severe_storm_disaster_count",
    "flood_disaster_count",
    "fire_disaster_count",
    "any_disaster_flag",
    "any_climate_disaster_flag",
]
county_year_disasters[DISASTER_ANNUAL_COLUMNS] = (
    county_year_disasters[DISASTER_ANNUAL_COLUMNS]
    .fillna(0)
    .astype(int)
)
county_year_disasters = (
    county_year_disasters.sort_values(["STCOFIPS", "Year"])
    .reset_index(drop=True)
)
county_year_disasters["cumulative_disaster_count"] = (
    county_year_disasters.groupby("STCOFIPS")["disaster_count"].cumsum()
)
county_year_disasters["cumulative_climate_disaster_count"] = (
    county_year_disasters.groupby("STCOFIPS")[
        "climate_disaster_count"
    ].cumsum()
)
county_year_disasters["recent_3yr_disaster_count"] = (
    county_year_disasters.groupby("STCOFIPS")["disaster_count"]
    .transform(lambda values: values.rolling(3, min_periods=1).sum())
)
county_year_disasters["recent_3yr_climate_disaster_count"] = (
    county_year_disasters.groupby("STCOFIPS")["climate_disaster_count"]
    .transform(lambda values: values.rolling(3, min_periods=1).sum())
)

DISASTER_HISTORY_COLUMNS = [
    "cumulative_disaster_count",
    "cumulative_climate_disaster_count",
    "recent_3yr_disaster_count",
    "recent_3yr_climate_disaster_count",
]
county_year_disasters[DISASTER_HISTORY_COLUMNS] = (
    county_year_disasters[DISASTER_HISTORY_COLUMNS].astype(int)
)

county_year_disaster_enhanced = county_year_spatial.merge(
    county_year_disasters,
    on=["STCOFIPS", "Year"],
    how="left",
    validate="one_to_one",
)

DISASTER_FEATURE_COLUMNS = (
    DISASTER_ANNUAL_COLUMNS + DISASTER_HISTORY_COLUMNS
)
if county_year_disaster_enhanced[
    DISASTER_FEATURE_COLUMNS
].isna().any().any():
    raise ValueError("Missing disaster indicators found after the merge.")

DISASTER_INDICATORS_PATH = (
    COUNTY_YEAR_DIR / "county_year_disaster_indicators.csv"
)
DISASTER_ENHANCED_PATH = (
    COUNTY_YEAR_DIR / "county_year_housing_spatial_disaster_features.csv"
)
county_year_disasters.to_csv(DISASTER_INDICATORS_PATH, index=False)
county_year_disaster_enhanced.to_csv(DISASTER_ENHANCED_PATH, index=False)

disaster_validation = pd.DataFrame(
    {
        "check": [
            "County-year rows",
            "Florida counties",
            "Start year",
            "End year",
            "Duplicate county-years",
            "Missing disaster values",
        ],
        "value": [
            len(county_year_disaster_enhanced),
            county_year_disaster_enhanced["STCOFIPS"].nunique(),
            county_year_disaster_enhanced["Year"].min(),
            county_year_disaster_enhanced["Year"].max(),
            int(
                county_year_disaster_enhanced.duplicated(
                    ["STCOFIPS", "Year"]
                ).sum()
            ),
            int(
                county_year_disaster_enhanced[DISASTER_FEATURE_COLUMNS]
                .isna()
                .sum()
                .sum()
            ),
        ],
    }
)

display(disaster_validation)


,check,value
0,County-year rows,1000
1,Florida counties,67
2,Start year,2011
3,End year,2025
4,Duplicate county-years,0
5,Missing disaster values,0


## 5. ACS Socioeconomic and Affordability Features

The workflow uses ACS five-year county estimates for 2011–2024. The cached raw
API response is used by default. Poverty, unemployment, and housing-tenure
rates are derived from ACS counts, then county-specific linear trends over
2020–2024 provide the documented 2025 estimates.


In [17]:
ACS_RAW_DIR = RAW_DATA / "acs"
ACS_RAW_DIR.mkdir(parents=True, exist_ok=True)
ACS_RAW_PATH = ACS_RAW_DIR / "acs_county_year_raw_2011_2024.csv"
ACS_CLEAN_PATH = COUNTY_YEAR_DIR / "acs_county_year_clean_2011_2024.csv"

ACS_VARIABLES = {
    "B01003_001E": "acs_total_population",
    "B19013_001E": "median_household_income",
    "B17001_001E": "poverty_universe",
    "B17001_002E": "poverty_count",
    "B23025_003E": "labor_force",
    "B23025_005E": "unemployed_count",
    "B25003_001E": "occupied_housing_units",
    "B25003_002E": "owner_occupied_units",
    "B25003_003E": "renter_occupied_units",
    "B25064_001E": "median_gross_rent",
    "B25077_001E": "acs_median_home_value",
}


def download_acs_data():
    import os
    import time
    from getpass import getpass

    import requests

    census_api_key = os.getenv("CENSUS_API_KEY") or getpass(
        "Enter your Census API key: "
    )
    yearly_frames = []

    for year in range(ACS_START_YEAR, ACS_END_YEAR + 1):
        response = requests.get(
            f"https://api.census.gov/data/{year}/acs/acs5",
            params={
                "get": "NAME," + ",".join(ACS_VARIABLES),
                "for": "county:*",
                "in": "state:12",
                "key": census_api_key,
            },
            timeout=30,
        )
        response.raise_for_status()
        payload = response.json()
        year_data = pd.DataFrame(payload[1:], columns=payload[0])
        year_data["Year"] = year
        year_data["STCOFIPS"] = (
            year_data["state"].astype(str).str.zfill(2)
            + year_data["county"].astype(str).str.zfill(3)
        )
        year_data = year_data.rename(columns=ACS_VARIABLES)
        yearly_frames.append(year_data)
        time.sleep(0.25)

    acs_raw = pd.concat(yearly_frames, ignore_index=True)
    numeric_columns = list(ACS_VARIABLES.values())
    acs_raw[numeric_columns] = acs_raw[numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    )
    return acs_raw[
        ["STCOFIPS", "Year", "NAME"] + numeric_columns
    ].copy()


if REFRESH_ACS_DATA:
    acs_raw = download_acs_data()
    acs_raw.to_csv(ACS_RAW_PATH, index=False)
else:
    if not ACS_RAW_PATH.exists():
        raise FileNotFoundError(
            "Cached ACS data are missing. Enable REFRESH_ACS_DATA to download."
        )
    acs_raw = pd.read_csv(ACS_RAW_PATH)

print(
    f"ACS snapshot: {acs_raw.shape}; "
    f"years {acs_raw['Year'].min()}–{acs_raw['Year'].max()}"
)


ACS snapshot: (938, 14); years 2011–2024


In [18]:
acs_clean = acs_raw.copy()
acs_clean["STCOFIPS"] = acs_clean["STCOFIPS"].astype(str).str.zfill(5)

ACS_NUMERIC_COLUMNS = list(ACS_VARIABLES.values())
acs_clean[ACS_NUMERIC_COLUMNS] = acs_clean[
    ACS_NUMERIC_COLUMNS
].apply(pd.to_numeric, errors="coerce")

acs_clean["poverty_rate"] = (
    acs_clean["poverty_count"] / acs_clean["poverty_universe"] * 100
)
acs_clean["unemployment_rate"] = (
    acs_clean["unemployed_count"] / acs_clean["labor_force"] * 100
)
acs_clean["owner_occupied_share"] = (
    acs_clean["owner_occupied_units"]
    / acs_clean["occupied_housing_units"]
    * 100
)
acs_clean["renter_occupied_share"] = (
    acs_clean["renter_occupied_units"]
    / acs_clean["occupied_housing_units"]
    * 100
)
acs_clean["housing_tenure_share_total"] = (
    acs_clean["owner_occupied_share"]
    + acs_clean["renter_occupied_share"]
)

ACS_DERIVED_COLUMNS = [
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
]

if len(acs_clean) != 938 or acs_clean["STCOFIPS"].nunique() != 67:
    raise ValueError("Unexpected ACS county-year coverage.")
if acs_clean.duplicated(["STCOFIPS", "Year"]).any():
    raise ValueError("Duplicate ACS county-year rows found.")
if acs_clean[ACS_NUMERIC_COLUMNS + ACS_DERIVED_COLUMNS].isna().any().any():
    raise ValueError("Missing ACS values found after cleaning.")

acs_clean.to_csv(ACS_CLEAN_PATH, index=False)

acs_validation = pd.DataFrame(
    {
        "check": [
            "Rows",
            "Florida counties",
            "Years",
            "Duplicate county-years",
            "Missing key values",
            "Minimum owner-renter total",
            "Maximum owner-renter total",
        ],
        "value": [
            len(acs_clean),
            acs_clean["STCOFIPS"].nunique(),
            acs_clean["Year"].nunique(),
            int(acs_clean.duplicated(["STCOFIPS", "Year"]).sum()),
            int(
                acs_clean[ACS_NUMERIC_COLUMNS + ACS_DERIVED_COLUMNS]
                .isna()
                .sum()
                .sum()
            ),
            acs_clean["housing_tenure_share_total"].min(),
            acs_clean["housing_tenure_share_total"].max(),
        ],
    }
)

display(acs_validation)


,check,value
0,Rows,938.0
1,Florida counties,67.0
2,Years,14.0
3,Duplicate county-years,0.0
4,Missing key values,0.0
5,Minimum owner-renter total,100.0
6,Maximum owner-renter total,100.0


In [19]:
county_year_disaster = pd.read_csv(DISASTER_ENHANCED_PATH)
county_year_disaster["STCOFIPS"] = (
    county_year_disaster["STCOFIPS"].astype(str).str.zfill(5)
)
acs_clean["STCOFIPS"] = acs_clean["STCOFIPS"].astype(str).str.zfill(5)

ACS_FEATURE_COLUMNS = [
    "STCOFIPS",
    "Year",
    "acs_total_population",
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
    "median_gross_rent",
    "acs_median_home_value",
]

county_year_acs = county_year_disaster.merge(
    acs_clean[ACS_FEATURE_COLUMNS],
    on=["STCOFIPS", "Year"],
    how="left",
    validate="one_to_one",
)
county_year_acs["price_to_income_ratio"] = (
    county_year_acs["avg_annual_housing_price"]
    / county_year_acs["median_household_income"]
)

ACS_PROJECTION_COLUMNS = [
    "acs_total_population",
    "median_household_income",
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
    "median_gross_rent",
    "acs_median_home_value",
]
PERCENTAGE_COLUMNS = {
    "poverty_rate",
    "unemployment_rate",
    "owner_occupied_share",
    "renter_occupied_share",
}
PROJECTION_BASE_YEARS = [2020, 2021, 2022, 2023, 2024]

county_year_acs["acs_estimated_2025_flag"] = 0

for county_fips in county_year_acs["STCOFIPS"].unique():
    county_mask = county_year_acs["STCOFIPS"].eq(county_fips)
    county_data = county_year_acs.loc[county_mask]

    for column in ACS_PROJECTION_COLUMNS:
        recent_data = county_data.loc[
            county_data["Year"].isin(PROJECTION_BASE_YEARS),
            ["Year", column],
        ].dropna()
        target_mask = (
            county_mask
            & county_year_acs["Year"].eq(2025)
            & county_year_acs[column].isna()
        )

        if target_mask.sum() != 1:
            continue

        if len(recent_data) >= 2:
            coefficients = np.polyfit(
                recent_data["Year"], recent_data[column], deg=1
            )
            projected_value = np.polyval(coefficients, 2025)
        else:
            projected_value = county_data.loc[
                county_data["Year"].eq(2024), column
            ].iloc[0]

        if column in PERCENTAGE_COLUMNS:
            projected_value = np.clip(projected_value, 0, 100)
        else:
            projected_value = max(projected_value, 0)

        county_year_acs.loc[target_mask, column] = projected_value

county_year_acs.loc[
    county_year_acs["Year"].eq(2025), "acs_estimated_2025_flag"
] = 1
county_year_acs["price_to_income_ratio"] = (
    county_year_acs["avg_annual_housing_price"]
    / county_year_acs["median_household_income"]
)

ACS_FINAL_COLUMNS = ACS_PROJECTION_COLUMNS + ["price_to_income_ratio"]
if county_year_acs[ACS_FINAL_COLUMNS].isna().any().any():
    raise ValueError("Missing ACS values remain after the 2025 projection.")

ACS_ENHANCED_PATH = (
    COUNTY_YEAR_DIR
    / "county_year_housing_spatial_disaster_acs_features.csv"
)
county_year_acs.to_csv(ACS_ENHANCED_PATH, index=False)

acs_merge_validation = pd.DataFrame(
    {
        "check": [
            "County-year rows",
            "Florida counties",
            "Projected 2025 rows",
            "Duplicate county-years",
            "Missing ACS values",
        ],
        "value": [
            len(county_year_acs),
            county_year_acs["STCOFIPS"].nunique(),
            int(county_year_acs["acs_estimated_2025_flag"].sum()),
            int(county_year_acs.duplicated(["STCOFIPS", "Year"]).sum()),
            int(county_year_acs[ACS_FINAL_COLUMNS].isna().sum().sum()),
        ],
    }
)

display(acs_merge_validation)


,check,value
0,County-year rows,1000
1,Florida counties,67
2,Projected 2025 rows,67
3,Duplicate county-years,0
4,Missing ACS values,0


## 6. NFIP Insurance-Loss Features

The selected NFIP claims endpoint is aggregated to county-year indicators.
County-years without claims receive zeros; cumulative and recent three-year
claim measures are then calculated. The frozen county-year snapshot is used by
default so ordinary reruns reproduce the study dataset exactly.


In [20]:
NFIP_RAW_DIR = RAW_DATA / "nfip"
NFIP_RAW_DIR.mkdir(parents=True, exist_ok=True)
NFIP_RAW_PATH = NFIP_RAW_DIR / "florida_nfip_claims_2011_2025.csv"
NFIP_CACHE_PATH = COUNTY_YEAR_DIR / "nfip_county_year_claim_indicators.csv"


def download_nfip_claims():
    import requests

    endpoint = "https://www.fema.gov/api/open/v2/FimaNfipClaims"
    selected_columns = [
        "state",
        "countyCode",
        "yearOfLoss",
        "dateOfLoss",
        "amountPaidOnBuildingClaim",
        "amountPaidOnContentsClaim",
        "amountPaidOnIncreasedCostOfComplianceClaim",
        "netBuildingPaymentAmount",
        "netContentsPaymentAmount",
        "netIccPaymentAmount",
        "totalBuildingInsuranceCoverage",
        "totalContentsInsuranceCoverage",
        "policyCount",
        "id",
    ]
    records = []
    batch_size = 5000
    skip = 0

    while True:
        response = requests.get(
            endpoint,
            params={
                "$filter": (
                    "state eq 'FL' and yearOfLoss ge 2011 "
                    "and yearOfLoss le 2025"
                ),
                "$select": ",".join(selected_columns),
                "$top": batch_size,
                "$skip": skip,
            },
            timeout=120,
        )
        response.raise_for_status()
        batch = response.json()["FimaNfipClaims"]
        if not batch:
            break
        records.extend(batch)
        if len(batch) < batch_size:
            break
        skip += batch_size

    return pd.DataFrame(records)


def build_nfip_county_year_features(claims, project_grid):
    claims = claims.dropna(subset=["countyCode"]).copy()
    claims["STCOFIPS"] = (
        claims["countyCode"]
        .astype(str)
        .str.replace(".0", "", regex=False)
        .str.zfill(5)
    )
    claims["Year"] = claims["yearOfLoss"].astype(int)

    numeric_columns = [
        "netBuildingPaymentAmount",
        "netContentsPaymentAmount",
        "netIccPaymentAmount",
        "totalBuildingInsuranceCoverage",
        "totalContentsInsuranceCoverage",
        "policyCount",
    ]
    claims[numeric_columns] = claims[numeric_columns].apply(
        pd.to_numeric, errors="coerce"
    )
    claims["nfip_total_claim_payment"] = (
        claims["netBuildingPaymentAmount"].fillna(0)
        + claims["netContentsPaymentAmount"].fillna(0)
        + claims["netIccPaymentAmount"].fillna(0)
    )

    aggregated = (
        claims.groupby(["STCOFIPS", "Year"])
        .agg(
            nfip_claim_count=("id", "count"),
            nfip_policy_count_sum=("policyCount", "sum"),
            nfip_total_building_payment=(
                "netBuildingPaymentAmount", "sum"
            ),
            nfip_total_contents_payment=(
                "netContentsPaymentAmount", "sum"
            ),
            nfip_total_icc_payment=("netIccPaymentAmount", "sum"),
            nfip_total_claim_payment=("nfip_total_claim_payment", "sum"),
            nfip_avg_claim_payment=("nfip_total_claim_payment", "mean"),
            nfip_total_building_coverage=(
                "totalBuildingInsuranceCoverage", "sum"
            ),
            nfip_total_contents_coverage=(
                "totalContentsInsuranceCoverage", "sum"
            ),
        )
        .reset_index()
    )

    nfip_features = project_grid.merge(
        aggregated,
        on=["STCOFIPS", "Year"],
        how="left",
        validate="one_to_one",
    )
    annual_columns = [
        column
        for column in nfip_features.columns
        if column not in {"STCOFIPS", "Year"}
    ]
    nfip_features[annual_columns] = nfip_features[annual_columns].fillna(0)
    nfip_features = nfip_features.sort_values(
        ["STCOFIPS", "Year"]
    ).reset_index(drop=True)
    nfip_features["nfip_cumulative_claim_count"] = (
        nfip_features.groupby("STCOFIPS")["nfip_claim_count"].cumsum()
    )
    nfip_features["nfip_cumulative_claim_payment"] = (
        nfip_features.groupby("STCOFIPS")[
            "nfip_total_claim_payment"
        ].cumsum()
    )
    nfip_features["nfip_recent_3yr_claim_count"] = (
        nfip_features.groupby("STCOFIPS")["nfip_claim_count"]
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )
    nfip_features["nfip_recent_3yr_claim_payment"] = (
        nfip_features.groupby("STCOFIPS")["nfip_total_claim_payment"]
        .rolling(3, min_periods=1)
        .sum()
        .reset_index(level=0, drop=True)
    )
    nfip_features["nfip_claim_year_indicator"] = (
        nfip_features["nfip_claim_count"].gt(0).astype(int)
    )
    return nfip_features


In [21]:
project_grid = county_year_acs[["STCOFIPS", "Year"]].drop_duplicates().copy()
project_grid["STCOFIPS"] = project_grid["STCOFIPS"].str.zfill(5)

if REFRESH_NFIP_DATA:
    nfip_claims = download_nfip_claims()
    nfip_claims.to_csv(NFIP_RAW_PATH, index=False)
    nfip_features = build_nfip_county_year_features(
        nfip_claims, project_grid
    )
    nfip_features.to_csv(NFIP_CACHE_PATH, index=False)
else:
    if not NFIP_CACHE_PATH.exists():
        raise FileNotFoundError(
            "Cached NFIP indicators are missing. Enable REFRESH_NFIP_DATA "
            "to download and rebuild them."
        )
    nfip_features = pd.read_csv(
        NFIP_CACHE_PATH, dtype={"STCOFIPS": str}
    )

nfip_features["STCOFIPS"] = nfip_features["STCOFIPS"].str.zfill(5)
nfip_features["Year"] = nfip_features["Year"].astype(int)

NFIP_FEATURE_COLUMNS = [
    "nfip_claim_count",
    "nfip_policy_count_sum",
    "nfip_total_building_payment",
    "nfip_total_contents_payment",
    "nfip_total_icc_payment",
    "nfip_total_claim_payment",
    "nfip_avg_claim_payment",
    "nfip_total_building_coverage",
    "nfip_total_contents_coverage",
    "nfip_cumulative_claim_count",
    "nfip_cumulative_claim_payment",
    "nfip_recent_3yr_claim_count",
    "nfip_recent_3yr_claim_payment",
    "nfip_claim_year_indicator",
]

if len(nfip_features) != 1000:
    raise ValueError("Expected 1,000 project-aligned NFIP rows.")
if nfip_features.duplicated(["STCOFIPS", "Year"]).any():
    raise ValueError("Duplicate NFIP county-year rows found.")
if nfip_features[NFIP_FEATURE_COLUMNS].isna().any().any():
    raise ValueError("Missing NFIP feature values found.")

county_year_features = county_year_acs.merge(
    nfip_features[["STCOFIPS", "Year"] + NFIP_FEATURE_COLUMNS],
    on=["STCOFIPS", "Year"],
    how="left",
    validate="one_to_one",
)

display(
    pd.DataFrame(
        {
            "check": [
                "NFIP county-year rows",
                "County-years with claims",
                "County-years without claims",
                "Missing NFIP values",
            ],
            "value": [
                len(nfip_features),
                int(nfip_features["nfip_claim_year_indicator"].sum()),
                int(
                    nfip_features["nfip_claim_year_indicator"].eq(0).sum()
                ),
                int(
                    nfip_features[NFIP_FEATURE_COLUMNS].isna().sum().sum()
                ),
            ],
        }
    )
)


,check,value
0,NFIP county-year rows,1000
1,County-years with claims,759
2,County-years without claims,241
3,Missing NFIP values,0


## 7. Final Integrated Dataset Validation and Export

The complete feature dataset is validated for study coverage, unique
county-year keys, expected column count, and missing values. Missingness is
retained only for structurally unavailable lag and acceleration indicators.


In [22]:
final_duplicate_count = int(
    county_year_features.duplicated(["STCOFIPS", "Year"]).sum()
)
final_missing = county_year_features.isna().sum()
expected_missing_columns = {
    "prev_year_housing_price",
    "annual_price_growth_dollar",
    "annual_price_growth_pct",
    "price_growth_acceleration",
    "high_growth_flag",
}
unexpected_missing_columns = set(
    final_missing.loc[final_missing.gt(0)].index
) - expected_missing_columns

if len(county_year_features) != 1000:
    raise ValueError("Expected 1,000 final county-year observations.")
if county_year_features.shape[1] != 75:
    raise ValueError("Expected 75 final feature columns.")
if county_year_features["STCOFIPS"].nunique() != 67:
    raise ValueError("Expected 67 Florida counties.")
if final_duplicate_count:
    raise ValueError("Duplicate county-year rows found in the final dataset.")
if unexpected_missing_columns:
    raise ValueError(
        "Unexpected missing values found in: "
        + ", ".join(sorted(unexpected_missing_columns))
    )

county_year_features.to_csv(FINAL_FEATURE_PATH, index=False)

final_validation = pd.DataFrame(
    {
        "check": [
            "Final rows",
            "Final columns",
            "Florida counties",
            "Start year",
            "End year",
            "Duplicate county-years",
            "Total expected structural missing values",
            "Unexpected missing columns",
        ],
        "value": [
            len(county_year_features),
            county_year_features.shape[1],
            county_year_features["STCOFIPS"].nunique(),
            county_year_features["Year"].min(),
            county_year_features["Year"].max(),
            final_duplicate_count,
            int(final_missing.sum()),
            len(unexpected_missing_columns),
        ],
    }
)

display(final_validation)
print(f"Saved: {FINAL_FEATURE_PATH.relative_to(PROJECT_ROOT)}")


,check,value
0,Final rows,1000
1,Final columns,75
2,Florida counties,67
3,Start year,2011
4,End year,2025
5,Duplicate county-years,0
6,Total expected structural missing values,82
7,Unexpected missing columns,0


Saved: data\processed\florida_county_year_features_2011_2025.csv
